# Hydro Lens — Microplastic Detector Training (40 Epochs)

**Model:** YOLOv8n · **Task:** binary object detection (`particle`) · **Epochs:** 40

This notebook is the single, self-contained training record for the hackathon demo.
Every artifact produced by training (weights, metrics, curves, logs) is written into
one folder: `training/hydro_lens_v1/`.

| Item | Value |
|------|-------|
| Dataset | D1 (`sombsuk/dataset_microplastic`) + D4 (`andreusv/Microplastic-Fragments-WWTP-Dataset`) |
| Splits | 3,709 train / 558 val / 301 test |
| Classes | 1 (`particle`) |
| Image size | 640 |
| Batch | 16 · Seed 42 · Patience 10 |
| Final val (ep 40) | P 0.88 · R 0.87 · mAP50 0.94 · mAP50-95 0.69 |
| Held-out test | P 0.773 · R 0.776 · mAP50 0.846 · mAP50-95 0.676 |

**Run order:** Dataset prep → Train (40 epochs) → Test evaluation → Curves.
Re-running is safe: the dataset step is skipped if `data/processed/` already exists,
and the run folder is reused (`exist_ok=True`).

In [ ]:
from pathlib import Path
import json, shutil, sys

import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

# notebook lives in training/, repo root is its parent
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

OUT_DIR = ROOT / "training" / "hydro_lens_v1"   # single folder for every train file
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = ROOT / "data" / "processed" / "data.yaml"
WEIGHTS = OUT_DIR / "weights" / "best.pt"

EPOCHS, BATCH, IMGSZ, SEED, PATIENCE = 40, 16, 640, 42, 10

print(f"repo:   {ROOT}")
print(f"artifacts -> {OUT_DIR}")

## 1 · Dataset preparation

Builds `data/processed/` (YOLO format, single class `particle`) from the raw D1 zip
archives and D4 parquet files via `src/data/prepare.py`. Skipped when already built.

In [ ]:
if DATA_YAML.exists():
    print(f"dataset already prepared: {DATA_YAML}")
else:
    sys.path.insert(0, str(ROOT))
    from src.data import prepare
    prepare.main()

## 2 · Train — 40 epochs

`project=training` + `name=hydro_lens_v1` sends **all** Ultralytics output (weights,
`results.csv`, `args.yaml`, confusion matrix, PR/F1 curves, batch predictions) into
`training/hydro_lens_v1/` — nothing is scattered anywhere else.

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    imgsz=IMGSZ,
    seed=SEED,
    device="cpu",          # use "mps" (Apple) / "cuda" (GPU) if available
    project=str(ROOT / "training"),
    name="hydro_lens_v1",
    exist_ok=True,
    verbose=True,
)

print("saved to", results.save_dir)

## 3 · Held-out test evaluation

Scores the best checkpoint on the 301-image test split (never seen during training)
and freezes the numbers as `test_metrics.json` in the run folder.

In [ ]:
best = YOLO(str(WEIGHTS))
metrics = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device="cpu", verbose=True)

test_metrics = {
    "precision": round(float(metrics.box.mp), 4),
    "recall": round(float(metrics.box.mr), 4),
    "mAP50": round(float(metrics.box.map50), 4),
    "mAP50-95": round(float(metrics.box.map), 4),
}

(OUT_DIR / "test_metrics.json").write_text(json.dumps(test_metrics, indent=2))
print(json.dumps(test_metrics, indent=2))

## 4 · Training curves

Rebuilt from `results.csv` so the plots live in the run folder next to the weights.

In [ ]:
df = pd.read_csv(OUT_DIR / "results.csv")
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(df["epoch"], df["train/box_loss"], label="box")
ax[0].plot(df["epoch"], df["val/box_loss"], label="val/box")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
ax[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
ax[1].set(title="Accuracy", xlabel="epoch"); ax[1].legend(); ax[1].grid(alpha=.3)

fig.tight_layout()
fig.savefig(OUT_DIR / "training_curves.png", dpi=150)
plt.show()

## 5 · Artifacts

Everything training produced, in one folder.

In [ ]:
shutil.copy2(DATA_YAML, OUT_DIR / "data.yaml")   # dataset config snapshot

for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(ROOT)}  ({p.stat().st_size / 1e6:.2f} MB)")

**Deployed weights:** `training/hydro_lens_v1/weights/best.pt` → copied to
`models/best.pt`, which the backend (`src/detect.py`) loads at startup.